# 01 — Data Ingestion

**Purpose:** Fetch raw macroeconomic data from all sources and save to `data/raw/`.

## Sources
- **FRED** — US series: GDP growth, CPI, PCE, fed funds rate, 2y/10y Treasury yields, unemployment, M2
- **World Bank** — Cross-country GDP growth, inflation, current account balance (via `wbdata`)
- **IMF** — WEO forecasts, global growth outlook (via `imf-reader`)

## Outputs
- `data/raw/fred_series.parquet`
- `data/raw/worldbank.parquet`
- `data/raw/imf_weo.parquet`

## Papermill Parameters
- `run_date` — ISO date string injected by the GitHub Actions workflow

In [ ]:
# Papermill parameters (injected at runtime)
run_date = None  # e.g. '2026-05-05'

In [ ]:
import os
import pandas as pd
import wbdata
import imf_reader
from fredapi import Fred
from pathlib import Path
from datetime import datetime

RAW_DIR = Path("data/raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)

# FRED key comes from GitHub Actions secret; set locally via environment variable
# PowerShell: $env:FRED_API_KEY = "your_key"
FRED_API_KEY = os.environ.get("FRED_API_KEY", "")
if not FRED_API_KEY:
    raise EnvironmentError("FRED_API_KEY not set. Add it as a repo secret or local env var.")

fred = Fred(api_key=FRED_API_KEY)
RUN_DATE = run_date or datetime.utcnow().strftime("%Y-%m-%d")
print(f"Run date: {RUN_DATE}")
print(f"Output directory: {RAW_DIR.resolve()}")

In [ ]:
# --- FRED ---
FRED_SERIES = {
    "gdp_growth":    "A191RL1Q225SBEA",  # Real GDP growth QoQ annualised
    "cpi_yoy":       "CPIAUCSL",          # CPI all items (index)
    "pce_yoy":       "PCEPI",             # PCE price index
    "fed_funds":     "FEDFUNDS",          # Fed funds effective rate
    "t2y":           "DGS2",              # 2-year Treasury yield
    "t10y":          "DGS10",             # 10-year Treasury yield
    "t3m":           "DGS3MO",            # 3-month Treasury yield
    "unrate":        "UNRATE",            # Unemployment rate
    "m2":            "M2SL",              # M2 money supply
    "credit_spread": "BAMLH0A0HYM2",      # ICE BofA HY OAS (credit spread)
}

frames = {}
for name, series_id in FRED_SERIES.items():
    try:
        s = fred.get_series(series_id)
        s.name = name
        frames[name] = s
        print(f"  {name}: {len(s)} obs, latest {s.index[-1].date()} = {s.iloc[-1]:.2f}")
    except Exception as e:
        print(f"  {name}: FAILED — {e}")

fred_df = pd.DataFrame(frames)
fred_df.index.name = "date"
fred_df.index = pd.to_datetime(fred_df.index)
fred_df = fred_df.sort_index()

out = RAW_DIR / "fred_series.parquet"
fred_df.to_parquet(out)
print(f"\nFRED saved: {fred_df.shape} → {out}")

In [ ]:
# --- World Bank ---
WB_INDICATORS = {
    "NY.GDP.MKTP.KD.ZG": "gdp_growth",
    "FP.CPI.TOTL.ZG":    "cpi_inflation",
    "BN.CAB.XOKA.GD.ZS": "current_account_pct_gdp",
    "GC.DOD.TOTL.GD.ZS": "govt_debt_pct_gdp",
    "SL.UEM.TOTL.ZS":    "unemployment_rate",
}

# G20 economies (ISO2 codes)
COUNTRIES = [
    "US", "CN", "DE", "JP", "GB", "FR", "IN", "BR", "CA", "AU",
    "KR", "MX", "ID", "TR", "SA", "ZA", "AR", "IT", "RU", "EU",
]

print("Fetching World Bank data...")
try:
    wb_df = wbdata.get_dataframe(WB_INDICATORS, country=COUNTRIES)
    wb_df = wb_df.rename(columns=WB_INDICATORS)
    wb_df.index.names = ["country", "date"]
    wb_df = wb_df.sort_index()

    out = RAW_DIR / "worldbank.parquet"
    wb_df.to_parquet(out)
    print(f"World Bank saved: {wb_df.shape} → {out}")
    print(wb_df.tail(5))
except Exception as e:
    print(f"World Bank fetch failed: {e}")

In [ ]:
# --- IMF World Economic Outlook ---
print("Fetching IMF WEO data...")
try:
    # imf_reader fetches from the IMF JSON:API
    # Available datasets: imf_reader.available_datasets()
    weo = imf_reader.fetch_data("WEO")

    # Keep key indicators for major economies
    WEO_INDICATORS = [
        "NGDP_RPCH",   # Real GDP growth
        "PCPIPCH",     # CPI inflation
        "LUR",         # Unemployment rate
        "BCA_NGDPD",   # Current account % GDP
        "GGXWDG_NGDP", # Govt gross debt % GDP
    ]
    MAJOR_ECONOMIES = [
        "United States", "China", "Germany", "Japan", "United Kingdom",
        "France", "India", "Brazil", "Canada", "Australia",
    ]

    mask_ind = weo["CONCEPT_CODE"].isin(WEO_INDICATORS)
    mask_cty = weo["COUNTRY"].isin(MAJOR_ECONOMIES)
    weo_df = weo[mask_ind & mask_cty].copy()

    out = RAW_DIR / "imf_weo.parquet"
    weo_df.to_parquet(out)
    print(f"IMF WEO saved: {weo_df.shape} → {out}")
    print(weo_df.head(5))
except Exception as e:
    print(f"IMF fetch failed: {e}")